## tl;dr

Four clients have current paid Square evidence: Samantha Bailey, Levy Family, Gabriella Scarpa, and Tyler Cooper. Catherine Squires is unresolved/paused, and Sanjay Roy has an owner-confirmed non-Square amount that could not be refreshed. The new policy prices can reproduce some accounts closely, but household-dinner semantics, delivery treatment, and unpriced kids/snack items make a single automatic backfill unsafe.

## Context & Methods

This notebook supports a pricing-architecture decision for Local Effort Cooperative. It normalizes observed charges to a four-week cycle and compares them with owner-supplied policy rates: breakfast $12/person, lunch $19/person, dinner $24/person, optional family dinner $45, 8% off food for four-week billing, and $10 weekly delivery. Membership dues and 4% store-credit rewards are excluded from meal totals.

### Key Assumptions

- The 8% four-week discount applies to food lines, not delivery or membership dues.
- A four-week billing cycle is used because the observed recurring invoices use 28-day cycles.
- Low/high scenarios preserve unresolved plan semantics instead of selecting a convenient answer.
- Square evidence was read on September 17, 2026. Repo plan snapshots are dated August 15-17, 2026.
- Gmail status did not return, and iMessage is unavailable; those channels are not treated as reconciled evidence.

## Data

Sources: live read-only Square recurring-invoice queries; `scripts/data/planner-meal-prep-billing-2026-08-17.json`; `scripts/weekly-order-seed-all.js`; `docs/weekly-order-levy-setup.md`; `scripts/correct-brain-operating-model-2026-09-15.mjs`; and Capital Master Record 2.4. Direct owner instructions in the current conversation supersede conflicting price figures in the master record for the proposed product price book.

In [1]:
from decimal import Decimal as D

def monthly_food(food_weekly, delivery_weekly=D('10')):
    return (D(str(food_weekly)) * D('0.92') + D(str(delivery_weekly))) * D('4')

rows = [
    {
        'client': 'Samantha Bailey',
        'status': 'confirmed current',
        'observed_4wk': D('440'),
        'standard_low': D('456'),
        'standard_high': D('496'),
        'basis': '3 lunches x 2 adults; pickup vs weekly delivery',
    },
    {
        'client': 'Levy Family',
        'status': 'confirmed current',
        'observed_4wk': D('1068'),
        'standard_low': D('1090.84'),
        'standard_high': D('1090.84'),
        'basis': 'discount-only sensitivity; kids meals/snacks lack policy prices',
    },
    {
        'client': 'Gabriella Scarpa',
        'status': 'confirmed current',
        'observed_4wk': D('1405'),
        'standard_low': monthly_food(D('225')),
        'standard_high': monthly_food(D('480')),
        'basis': '5 dinners for 4 adults; flat-family vs per-adult mode',
    },
    {
        'client': 'Tyler Cooper',
        'status': 'confirmed current; composition provisional',
        'observed_4wk': D('1088'),
        'standard_low': monthly_food(D('282')),
        'standard_high': monthly_food(D('345')),
        'basis': '8 breakfasts, 6 lunches, 3 dinners; dinner portion vs family-of-two mode',
    },
    {
        'client': 'Catherine Squires',
        'status': 'paused/unresolved',
        'observed_4wk': D('480'),
        'standard_low': D('400'),
        'standard_high': D('616'),
        'basis': '2 dinners for 3 adults; flat-family vs per-adult mode',
    },
    {
        'client': 'Sanjay Roy',
        'status': 'current status unresolved; non-Square amount',
        'observed_4wk': D('512'),
        'standard_low': monthly_food(D('152')),
        'standard_high': monthly_food(D('192')),
        'basis': '8 beef bowls; lunch vs dinner classification, with delivery',
    },
]

for row in rows:
    row['delta_vs_low'] = row['observed_4wk'] - row['standard_low']
    row['delta_vs_high'] = row['observed_4wk'] - row['standard_high']
    row['pct_vs_low'] = row['delta_vs_low'] / row['standard_low']
    row['pct_vs_high'] = row['delta_vs_high'] / row['standard_high']


In [2]:
print('| Client | Status | Observed / 4 weeks | Policy scenario | Difference |')
print('|---|---|---:|---:|---:|')
for row in rows:
    scenario = f"${row['standard_low']:,.2f}" if row['standard_low'] == row['standard_high'] else f"${row['standard_low']:,.2f}-${row['standard_high']:,.2f}"
    delta = f"${row['delta_vs_low']:+,.2f}" if row['delta_vs_low'] == row['delta_vs_high'] else f"${row['delta_vs_low']:+,.2f} to ${row['delta_vs_high']:+,.2f}"
    print(f"| {row['client']} | {row['status']} | ${row['observed_4wk']:,.2f} | {scenario} | {delta} |")

| Client | Status | Observed / 4 weeks | Policy scenario | Difference |
|---|---|---:|---:|---:|
| Samantha Bailey | confirmed current | $440.00 | $456.00-$496.00 | $-16.00 to $-56.00 |
| Levy Family | confirmed current | $1,068.00 | $1,090.84 | $-22.84 |
| Gabriella Scarpa | confirmed current | $1,405.00 | $868.00-$1,806.40 | $+537.00 to $-401.40 |
| Tyler Cooper | confirmed current; composition provisional | $1,088.00 | $1,077.76-$1,309.60 | $+10.24 to $-221.60 |
| Catherine Squires | paused/unresolved | $480.00 | $400.00-$616.00 | $+80.00 to $-136.00 |
| Sanjay Roy | current status unresolved; non-Square amount | $512.00 | $599.36-$746.56 | $-87.36 to $-234.56 |


In [3]:
confirmed = [row for row in rows if row['status'].startswith('confirmed current')]
print('Confirmed-current observed four-week total:', f"${sum(row['observed_4wk'] for row in confirmed):,.2f}")
print('Confirmed-current accounts:', len(confirmed))
assert monthly_food(D('225')) == D('868.00')
assert monthly_food(D('480')) == D('1806.40')
assert monthly_food(D('282')) == D('1077.76')
assert monthly_food(D('345')) == D('1309.60')

Confirmed-current observed four-week total: $4,001.00
Confirmed-current accounts: 4


In [4]:
import sqlite3

event_sql = """
WITH inputs(example, short_label, estimate_total, basis) AS (
  SELECT 'Offsite buffet, 20-30 guests', 'Offsite buffet', 45 * 30, '$45 x 30'
  UNION ALL SELECT 'Foodist buffet, 20-30 guests', 'Foodist buffet', 45 * 30 + 150, '$45 x 30 + $150'
  UNION ALL SELECT 'Firehouse buffet, 20-30 guests', 'Firehouse buffet', 45 * 30 + 750, '$45 x 30 + $750'
  UNION ALL SELECT 'Firehouse, 2 nights, 2 dinners, 16 lunches', 'Firehouse package', 1500 + 1000 + (8 * 2 * 19), '$1,500 + $1,000 + $304'
)
SELECT example, short_label, estimate_total, ROUND(estimate_total * 0.20, 2) AS deposit, basis
FROM inputs
"""
event_rows = sqlite3.connect(':memory:').execute(event_sql).fetchall()
event_rows

[('Offsite buffet, 20-30 guests', 'Offsite buffet', 1350, 270.0, '$45 x 30'),
 ('Foodist buffet, 20-30 guests',
  'Foodist buffet',
  1500,
  300.0,
  '$45 x 30 + $150'),
 ('Firehouse buffet, 20-30 guests',
  'Firehouse buffet',
  2100,
  420.0,
  '$45 x 30 + $750'),
 ('Firehouse, 2 nights, 2 dinners, 16 lunches',
  'Firehouse package',
  2804,
  560.8,
  '$1,500 + $1,000 + $304')]

In [5]:
meal_prep_sql = """
WITH client_pricing(client, status, observed_4wk, standard_low, standard_high, delta_vs_low, delta_vs_high, policy_display, difference_display, basis) AS (
  VALUES
    ('Samantha Bailey', 'Confirmed current', 440.00, 456.00, 496.00, -16.00, -56.00, '$456-$496', '-$16 to -$56', '3 lunches x 2 adults; pickup vs weekly delivery'),
    ('Levy Family', 'Confirmed current; sensitivity only', 1068.00, 1090.84, 1090.84, -22.84, -22.84, '$1,090.84', '-$22.84', 'Discount-only scenario; kids meals/snacks lack policy prices'),
    ('Gabriella Scarpa', 'Confirmed current', 1405.00, 868.00, 1806.40, 537.00, -401.40, '$868-$1,806.40', '+$537 to -$401.40', '5 dinners for 4 adults; flat-family vs per-adult mode'),
    ('Tyler Cooper', 'Confirmed current; composition provisional', 1088.00, 1077.76, 1309.60, 10.24, -221.60, '$1,077.76-$1,309.60', '+$10.24 to -$221.60', '8 breakfasts, 6 lunches, 3 dinners; portion vs family-of-two mode'),
    ('Catherine Squires', 'Paused/unresolved', 480.00, 400.00, 616.00, 80.00, -136.00, '$400-$616', '+$80 to -$136', '2 dinners for 3 adults; flat-family vs per-adult mode'),
    ('Sanjay Roy', 'Current status unresolved; non-Square amount', 512.00, 599.36, 746.56, -87.36, -234.56, '$599.36-$746.56', '-$87.36 to -$234.56', '8 beef bowls; lunch vs dinner classification, with delivery')
)
SELECT * FROM client_pricing
"""
meal_prep_sql_rows = sqlite3.connect(':memory:').execute(meal_prep_sql).fetchall()
assert len(meal_prep_sql_rows) == 6
meal_prep_sql_rows

[('Samantha Bailey',
  'Confirmed current',
  440.0,
  456.0,
  496.0,
  -16.0,
  -56.0,
  '$456-$496',
  '-$16 to -$56',
  '3 lunches x 2 adults; pickup vs weekly delivery'),
 ('Levy Family',
  'Confirmed current; sensitivity only',
  1068.0,
  1090.84,
  1090.84,
  -22.84,
  -22.84,
  '$1,090.84',
  '-$22.84',
  'Discount-only scenario; kids meals/snacks lack policy prices'),
 ('Gabriella Scarpa',
  'Confirmed current',
  1405.0,
  868.0,
  1806.4,
  537.0,
  -401.4,
  '$868-$1,806.40',
  '+$537 to -$401.40',
  '5 dinners for 4 adults; flat-family vs per-adult mode'),
 ('Tyler Cooper',
  'Confirmed current; composition provisional',
  1088.0,
  1077.76,
  1309.6,
  10.24,
  -221.6,
  '$1,077.76-$1,309.60',
  '+$10.24 to -$221.60',
  '8 breakfasts, 6 lunches, 3 dinners; portion vs family-of-two mode'),
 ('Catherine Squires',
  'Paused/unresolved',
  480.0,
  400.0,
  616.0,
  80.0,
  -136.0,
  '$400-$616',
  '+$80 to -$136',
  '2 dinners for 3 adults; flat-family vs per-adult mode'),


## Results

The strongest finding is structural: price differences cannot be interpreted as discounts until each plan records meal classification, portion/household mode, delivery inclusion, billing cadence, and eligible discount base. The policy is already close to Tyler's current invoice only under an individual-portion dinner interpretation; it is materially below or above several other plans depending on family-dinner semantics. Levy cannot be fully repriced until kids-meal and snack products have prices.

## Takeaways

1. Import current client arrangements as effective-dated agreements, not as global price rules.
2. Require an explicit pricing mode on every dinner line (`per_person`, `family_flat`, or negotiated override).
3. Store delivery, monthly discount, membership, and store-credit accrual as separate components.
4. Do not label residual differences as negotiated discounts until Gmail/iMessage or owner confirmation supplies a reason.